# Week-1 Security Lab — Objectives

- Implement and test **row-level** and **column-level** access controls (simulated locally)
- Demonstrate **dynamic data masking** patterns and **sensitivity labeling** best practices
- Verify access scenarios via simple tests and example queries

---

## Prerequisites

- Access to a Microsoft Fabric trial tenant (recommended for production-level testing)
- Local dev: Python 3.8+, pandas, pytest (the notebook uses small synthetic data to simulate behavior)
- Check `DP-700-Study-Plan.md` for the mapping to the "Manage a Microsoft Fabric environment" learning path

In [ ]:
### Setup & Imports

```python
# Optional (uncomment to install in notebook environment)
# !pip install pandas pytest

import pandas as pd
import numpy as np
import pathlib

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

def save_csv(df: pd.DataFrame, path: str) -> None:
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

print('Setup complete')
```

In [ ]:
# Data creation: users and transactions (synthetic)
users = pd.DataFrame([
    {"user_id": 1, "name": "Alice", "role": "analyst"},
    {"user_id": 2, "name": "Bob", "role": "data_engineer"},
    {"user_id": 3, "name": "Charlie", "role": "manager"}
])

transactions = pd.DataFrame({
    "tx_id": np.arange(1, 11),
    "user_id": np.random.choice(users.user_id, size=10),
    "amount": np.round(np.random.uniform(10, 1000, 10), 2),
    "ssn": ["123-45-6789", "987-65-4321", "111-22-3333", "444-55-6666", "222-33-4444", "555-66-7777", "666-77-8888", "777-88-9999", "888-99-0000", "999-00-1111"]
})

save_csv(users, 'labs/week1/data/users.csv')
save_csv(transactions, 'labs/week1/data/transactions.csv')

users.head(), transactions.head()


In [ ]:
# Core implementation: simple RLS and masking helpers
from typing import List

def apply_row_level_filter(df: pd.DataFrame, user_id: int, allowed_user_ids: List[int]) -> pd.DataFrame:
    """Simulate row-level security by filtering rows according to allowed_user_ids"""
    return df[df.user_id.isin(allowed_user_ids)]


def mask_ssn(value: str) -> str:
    """Simple dynamic mask example: show only last 4 digits"""
    return 'XXX-XX-' + value.split('-')[-1]

# Demonstration
filtered = apply_row_level_filter(transactions, user_id=1, allowed_user_ids=[1])
filtered['ssn_masked'] = filtered['ssn'].apply(mask_ssn)
filtered.head()


In [ ]:
# Tests & verification (pytest-style checks)

def test_row_filter():
    df = pd.DataFrame({"user_id": [1,2,3], "val": [10,20,30]})
    out = apply_row_level_filter(df, user_id=1, allowed_user_ids=[1])
    assert out.user_id.nunique() == 1


def test_mask_ssn():
    assert mask_ssn('123-45-6789') == 'XXX-XX-6789'

print('Run `%pytest` or call test functions manually in cells to verify behavior')
